# EduAI Assistant — Flashcard Generator


---

In [ ]:
!pip install pdfplumber spacy transformers torch sentencepiece groq -q
!python -m spacy download en_core_web_sm -q

import spacy
import re
from collections import Counter
print('All dependencies installed!')
print(f'spaCy version: {spacy.__version__}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All dependencies installed!
spaCy version: 3.8.11


## Upload and Process PDF files

In [ ]:
import pdfplumber

def extract_text_from_pdf(file_path):
    full_text = ''
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + '\n'
    return full_text

def clean_text(text):
    """Remove watermarks, URLs, emails, page numbers"""
    lines = text.split('\n')
    cleaned = []
    noise_patterns = ['whatsapp:', 'megalecture', 'mega lecture', 'email:',
                      'www.youtube', 'www.megalecture', 'youtube.com']

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        lower = stripped.lower()
        if any(p in lower for p in noise_patterns):
            continue
        if stripped.startswith('http') or stripped.startswith('www.'):
            continue
        if re.match(r'^\d{1,3}$', stripped):
            continue
        cleaned.append(stripped)

    text = '\n'.join(cleaned)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'[\+]?\d{1,3}[\s\-]?\d{2,4}[\s\-]?\d{3,4}[\s\-]?\d{3,4}', '', text)
    text = re.sub(r'(?i)whatsapp\s*:\s*', '', text)
    text = re.sub(r'(?i)page\s+\d+\s+of\s+\d+', '', text)
    text = re.sub(r'(?i)mega\s*lecture', '', text)
    text = re.sub(r'(?i)refined\s+by\s+\w+', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# Upload PDF
from google.colab import files
print('Upload a PDF file:')
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

raw_text = extract_text_from_pdf(file_name)
cleaned_text = clean_text(raw_text)

print(f'\nRaw text: {len(raw_text)} chars')
print(f'Cleaned text: {len(cleaned_text)} chars')
print(f'Noise removed: {len(raw_text) - len(cleaned_text)} chars')
print(f'\nFirst 500 chars of cleaned text:')
print(cleaned_text[:500])

In [ ]:
nlp = spacy.load('en_core_web_sm')

# Process text with spaCy
doc = nlp(cleaned_text[:50000])  # Limit to avoid memory issues

# Extract named entities
print('=' * 60)
print('  NAMED ENTITY RECOGNITION (NER) RESULTS')
print('=' * 60)

entity_types = {}
for ent in doc.ents:
    if ent.label_ not in entity_types:
        entity_types[ent.label_] = []
    entity_types[ent.label_].append(ent.text)

for label, entities in sorted(entity_types.items()):
    unique = list(set(entities))[:10]
    print(f'\n{label} ({len(entities)} found):')
    for e in unique:
        print(f'  - {e}')

print(f'\nTotal entities found: {len(doc.ents)}')
print(f'Unique entity types: {len(entity_types)}')

In [ ]:
STOP_ENTITIES = {
    'whatsapp', 'email', 'gmail', 'megalecture', 'youtube',
    'http', 'www', 'com', 'page', 'copyright', 'refined',
    'the', 'this', 'that', 'which', 'they', 'them',
    'what', 'where', 'when', 'how', 'who', 'why',
    'also', 'just', 'very', 'much', 'many', 'some',
}

def is_valid_term(term):
    lower = term.lower().strip()
    if len(lower) < 3:
        return False
    for stop in STOP_ENTITIES:
        if stop in lower:
            return False
    if re.match(r'^[\d\s\.\-\+]+$', lower):
        return False
    return True

# Extract noun chunks
noun_chunks = []
for chunk in doc.noun_chunks:
    text = chunk.text.strip()
    if len(text) > 3 and len(text.split()) <= 5 and is_valid_term(text):
        noun_chunks.append(text)

# Combine NER entities + noun chunks and count frequency
valid_entities = [ent.text.strip() for ent in doc.ents if is_valid_term(ent.text)]
all_terms = valid_entities + noun_chunks
term_counts = Counter(all_terms)

# Get top terms
top_terms = []
seen = set()
for term, count in term_counts.most_common(50):
    if count >= 2 and term.lower() not in seen:
        seen.add(term.lower())
        top_terms.append((term, count))

print('=' * 60)
print('  TOP KEY TERMS (for flashcard generation)')
print('=' * 60)
for i, (term, count) in enumerate(top_terms[:30]):
    print(f'  {i+1:2d}. {term:40s} (mentioned {count}x)')

In [ ]:
#Using FLAN-T5
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def get_context(text, term, window=300):
    idx = text.lower().find(term.lower())
    if idx == -1:
        return ''
    start = max(0, idx - window)
    end = min(len(text), idx + len(term) + window)
    return text[start:end].strip()

def generate_flan_t5(prompt, max_length=150):
    inputs = tokenizer(prompt, max_length=512, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, num_beams=4,
                                  no_repeat_ngram_size=3, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Generate flashcards for top 10 terms
print('\n')
print('  FLASHCARDS GENERATED BY FLAN-T5 (LOCAL)')

flan_flashcards = []
for term, count in top_terms[:10]:
    context = get_context(cleaned_text, term)
    if not context:
        continue

    definition = generate_flan_t5(f"Define '{term}' based on this context:\n{context[:500]}\nDefinition:")

    if len(definition.strip()) > 10:
        flan_flashcards.append({
            'type': 'definition', 'difficulty': 'easy',
            'front': f'What is {term}?', 'back': definition.strip()
        })
        print(f'\n Q: What is {term}?')
        print(f'   A: {definition.strip()}')

print(f'\nTotal FLAN-T5 flashcards: {len(flan_flashcards)}')

## FALLBACK METHOD: Groq

In [ ]:
GROQ_API_KEY = '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'
groq_flashcards = []

if GROQ_API_KEY:
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)

    prompt = f"""Based on the following educational text, generate 20 study flashcards.

Create a mix of:
- Definition cards (easy): "What is X?" with clear definitions
- Concept cards (medium): explain relationships and significance
- Example cards (hard): application and deeper understanding

Text:
{cleaned_text[:5000]}

Return flashcards in this format, one per line:
TYPE|DIFFICULTY|FRONT|BACK

Generate 20 flashcards:"""

    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': 'You are an expert educational content creator. Generate high-quality study flashcards.'},
            {'role': 'user', 'content': prompt}
        ],
        max_tokens=2000,
        temperature=0.3,
    )

    result = response.choices[0].message.content

    print('=' * 60)
    print('  FLASHCARDS GENERATED BY GROQ (CLOUD)')
    print('=' * 60)

    for line in result.strip().split('\n'):
        line = line.strip()
        if '|' not in line:
            continue
        parts = line.split('|')
        if len(parts) >= 4:
            card_type = parts[0].strip().lower()
            difficulty = parts[1].strip().lower()
            front = parts[2].strip()
            back = '|'.join(parts[3:]).strip()

            if front and back and len(front) > 5:
                groq_flashcards.append({
                    'type': card_type, 'difficulty': difficulty,
                    'front': front, 'back': back
                })
                emoji = {'easy': 'E', 'medium': 'M', 'hard': 'H'}.get(difficulty)
                print(f'\n{emoji} [{card_type.upper()}] Q: {front}')
                print(f'   A: {back}')

    print(f'\nTotal Groq flashcards: {len(groq_flashcards)}')
else:
    print('Groq API key wrong')


  FLASHCARDS GENERATED BY GROQ (CLOUD)

E [DEFINITION] Q: What is Vishwokarma Labs?
   A: Vishwokarma Labs is a technology-driven organization working in Robotics, Artificial Intelligence, and Internet of Things.

M [CONCEPT] Q: What is the purpose of Vishwokarma Labs TechX Kathmandu?
   A: The event aims to bring together students, creators, and innovators to present real-world robotics and IoT-based projects, demonstrate practical solutions, and engage in an interactive environment of learning and collaboration.

H [EXAMPLE] Q: What are the benefits of participating in Vishwokarma Labs TechX Kathmandu for students?
   A: Students can present their innovative projects, receive feedback from mentors and industry professionals, engage with like-minded peers, compete for recognition and cash prizes, and gain exposure to emerging technologies.

E [DEFINITION] Q: What is the date of Vishwokarma Labs TechX Kathmandu 2026?
   A: The event is scheduled to take place on 4 April 2026.

M [CONCE